[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/volume2_chapter_02/notebook_2A_data_poisoning.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 2A: Data Poisoning Attacks and Robust Training

**Volume 2, Chapter 2: Security, Privacy, and Robustness**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Implement common data poisoning attacks (label flipping, feature manipulation)
2. Understand how poisoning affects model performance on targeted subgroups
3. Apply outlier detection methods to identify poisoned samples
4. Implement robust training techniques that mitigate poisoning attacks
5. Evaluate the trade-offs between model accuracy and poisoning resistance

## Clinical Context

Data poisoning attacks represent a serious threat to healthcare AI. An attacker with access to training data can subtly modify labels or features to cause the resulting model to make targeted errors. In healthcare, this could mean:

- Missing diagnoses for specific patient populations
- Incorrect treatment recommendations for targeted individuals
- Systematic bias that degrades care quality

Understanding these attacks is essential for building robust defenses.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, 
    confusion_matrix, classification_report
)
from sklearn.neighbors import LocalOutlierFactor
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

COLORS = {
    'clean': '#27AE60',
    'poisoned': '#E74C3C',
    'detected': '#F39C12',
    'target': '#8E44AD'
}

print("✓ Libraries imported successfully")

---

## 1. Generate Healthcare Dataset

We create a simulated sepsis prediction dataset with clear subgroups that can be targeted by poisoning attacks.

In [ ]:
def generate_healthcare_data(n_samples=3000, seed=42):
    """
    Generate a simulated healthcare dataset for sepsis prediction.
    Includes demographic subgroups that can be targeted.
    """
    np.random.seed(seed)
    
    # Demographics
    age = np.random.normal(65, 15, n_samples)
    age = np.clip(age, 18, 95)
    
    # Create age groups for targeting
    age_group = pd.cut(age, bins=[0, 50, 70, 100], labels=['young', 'middle', 'elderly'])
    
    # Binary sex (for simplicity)
    sex = np.random.binomial(1, 0.5, n_samples)  # 0=female, 1=male
    
    # Clinical features
    heart_rate = np.random.normal(85, 18, n_samples)
    resp_rate = np.random.normal(18, 5, n_samples)
    temperature = np.random.normal(37.2, 0.8, n_samples)
    sbp = np.random.normal(120, 22, n_samples)
    wbc = np.random.normal(9.0, 4.0, n_samples)
    lactate = np.random.exponential(1.5, n_samples)
    creatinine = np.random.exponential(1.2, n_samples)
    
    # Comorbidities
    diabetes = np.random.binomial(1, 0.25, n_samples)
    hypertension = np.random.binomial(1, 0.35, n_samples)
    
    # Generate sepsis outcome based on features
    logit = (
        -4.5 +
        0.025 * (age - 65) +
        0.03 * (heart_rate - 85) +
        0.08 * (resp_rate - 18) +
        0.6 * (temperature - 37.2) +
        -0.015 * (sbp - 120) +
        0.08 * (wbc - 9.0) +
        0.35 * lactate +
        0.25 * creatinine +
        0.3 * diabetes +
        0.2 * hypertension
    )
    
    prob_sepsis = 1 / (1 + np.exp(-logit))
    sepsis = (np.random.random(n_samples) < prob_sepsis).astype(int)
    
    df = pd.DataFrame({
        'age': age,
        'age_group': age_group,
        'sex': sex,
        'heart_rate': heart_rate,
        'resp_rate': resp_rate,
        'temperature': temperature,
        'sbp': sbp,
        'wbc': wbc,
        'lactate': lactate,
        'creatinine': creatinine,
        'diabetes': diabetes,
        'hypertension': hypertension,
        'sepsis': sepsis
    })
    
    return df

# Generate clean dataset
df_clean = generate_healthcare_data(n_samples=3000)

print(f"Dataset size: {len(df_clean)} patients")
print(f"Sepsis prevalence: {df_clean['sepsis'].mean():.1%}")
print(f"\nSepsis by age group:")
print(df_clean.groupby('age_group')['sepsis'].agg(['mean', 'count']))

---

## 2. Data Poisoning Attacks

### 2.1 Label Flipping Attack

The simplest poisoning attack changes labels for targeted samples. We simulate an attacker who wants to reduce model sensitivity for elderly patients.

In [ ]:
def label_flipping_attack(df, target_condition, flip_rate=0.3, seed=42):
    """
    Perform label flipping attack on targeted samples.
    
    Parameters:
    -----------
    df : DataFrame
        Original dataset
    target_condition : callable
        Function that returns boolean mask for target samples
    flip_rate : float
        Proportion of positive labels in target group to flip
    
    Returns:
    --------
    DataFrame with poisoned labels, indices of poisoned samples
    """
    np.random.seed(seed)
    df_poisoned = df.copy()
    
    # Identify target samples with positive labels
    target_mask = target_condition(df)
    positive_targets = df.index[(target_mask) & (df['sepsis'] == 1)]
    
    # Select samples to flip
    n_to_flip = int(len(positive_targets) * flip_rate)
    flip_indices = np.random.choice(positive_targets, n_to_flip, replace=False)
    
    # Flip labels (1 -> 0)
    df_poisoned.loc[flip_indices, 'sepsis'] = 0
    
    print(f"Label Flipping Attack Summary:")
    print(f"  Target group size: {target_mask.sum()}")
    print(f"  Positive cases in target: {len(positive_targets)}")
    print(f"  Labels flipped: {n_to_flip} ({flip_rate:.0%} of positive targets)")
    print(f"  Overall poisoning rate: {n_to_flip/len(df):.1%} of dataset")
    
    return df_poisoned, flip_indices

# Attack targeting elderly patients (age > 70)
target_elderly = lambda df: df['age_group'] == 'elderly'

df_poisoned_labels, poisoned_indices = label_flipping_attack(
    df_clean, 
    target_condition=target_elderly,
    flip_rate=0.4
)

# Compare label distributions
print("\nSepsis prevalence comparison:")
print(f"  Clean data (elderly): {df_clean[df_clean['age_group']=='elderly']['sepsis'].mean():.1%}")
print(f"  Poisoned data (elderly): {df_poisoned_labels[df_poisoned_labels['age_group']=='elderly']['sepsis'].mean():.1%}")

### 2.2 Feature Manipulation Attack

More sophisticated attacks modify features rather than labels. This can be harder to detect because labels remain correct.

In [ ]:
def feature_manipulation_attack(df, target_condition, features_to_modify, 
                                 modification_strength=0.5, poison_rate=0.2, seed=42):
    """
    Modify features for targeted samples to weaken learned associations.
    
    Parameters:
    -----------
    df : DataFrame
        Original dataset
    target_condition : callable
        Function identifying target samples
    features_to_modify : list
        Features to perturb
    modification_strength : float
        Strength of perturbation (in std deviations)
    poison_rate : float
        Proportion of target samples to modify
    """
    np.random.seed(seed)
    df_poisoned = df.copy()
    
    # Identify target samples
    target_mask = target_condition(df)
    target_indices = df.index[target_mask]
    
    # Select samples to poison
    n_to_poison = int(len(target_indices) * poison_rate)
    poison_indices = np.random.choice(target_indices, n_to_poison, replace=False)
    
    # Modify features
    for feature in features_to_modify:
        feature_std = df[feature].std()
        # Add noise that pushes features toward "healthy" values
        perturbation = np.random.normal(0, feature_std * modification_strength, n_to_poison)
        df_poisoned.loc[poison_indices, feature] += perturbation
    
    print(f"Feature Manipulation Attack Summary:")
    print(f"  Target group size: {target_mask.sum()}")
    print(f"  Samples modified: {n_to_poison}")
    print(f"  Features modified: {features_to_modify}")
    print(f"  Modification strength: {modification_strength} std deviations")
    
    return df_poisoned, poison_indices

# Attack: modify vital signs for elderly sepsis patients
target_elderly_sepsis = lambda df: (df['age_group'] == 'elderly') & (df['sepsis'] == 1)

df_poisoned_features, poisoned_feature_indices = feature_manipulation_attack(
    df_clean,
    target_condition=target_elderly_sepsis,
    features_to_modify=['heart_rate', 'resp_rate', 'lactate', 'wbc'],
    modification_strength=0.6,
    poison_rate=0.5
)

---

## 3. Impact of Poisoning on Model Performance

Let's train models on clean and poisoned data to see how attacks affect performance.

In [ ]:
def prepare_data(df, feature_cols, test_size=0.3):
    """Prepare data for training."""
    X = df[feature_cols].values
    y = df['sepsis'].values
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

def train_and_evaluate(X_train, X_test, y_train, y_test, model_name='Model'):
    """Train logistic regression and evaluate."""
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    
    metrics = {
        'auroc': roc_auc_score(y_test, y_pred_prob),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred)
    }
    
    return model, metrics

# Define feature columns
feature_cols = ['age', 'sex', 'heart_rate', 'resp_rate', 'temperature', 
                'sbp', 'wbc', 'lactate', 'creatinine', 'diabetes', 'hypertension']

# Prepare and train on clean data
X_train_clean, X_test, y_train_clean, y_test, scaler = prepare_data(df_clean, feature_cols)
model_clean, metrics_clean = train_and_evaluate(X_train_clean, X_test, y_train_clean, y_test)

# Prepare and train on label-poisoned data
X_train_lp, _, y_train_lp, _, _ = prepare_data(df_poisoned_labels, feature_cols)
model_poisoned_label, metrics_poisoned_label = train_and_evaluate(X_train_lp, X_test, y_train_lp, y_test)

# Prepare and train on feature-poisoned data  
X_train_fp, _, y_train_fp, _, _ = prepare_data(df_poisoned_features, feature_cols)
model_poisoned_feature, metrics_poisoned_feature = train_and_evaluate(X_train_fp, X_test, y_train_fp, y_test)

print("Overall Model Performance Comparison")
print("=" * 60)
results = pd.DataFrame({
    'Clean Data': metrics_clean,
    'Label Poisoning': metrics_poisoned_label,
    'Feature Poisoning': metrics_poisoned_feature
}).T
print(results.round(3))

In [ ]:
def evaluate_subgroup(model, scaler, df_test, feature_cols, subgroup_mask, subgroup_name):
    """Evaluate model on specific subgroup."""
    df_subgroup = df_test[subgroup_mask]
    if len(df_subgroup) == 0 or df_subgroup['sepsis'].sum() == 0:
        return None
    
    X = scaler.transform(df_subgroup[feature_cols].values)
    y = df_subgroup['sepsis'].values
    
    y_pred_prob = model.predict_proba(X)[:, 1]
    y_pred = model.predict(X)
    
    return {
        'subgroup': subgroup_name,
        'n_samples': len(df_subgroup),
        'n_positive': y.sum(),
        'auroc': roc_auc_score(y, y_pred_prob) if y.sum() > 0 and y.sum() < len(y) else np.nan,
        'recall': recall_score(y, y_pred)
    }

# Create test set with subgroup info
df_test_full = df_clean.iloc[int(len(df_clean)*0.7):].copy()

# Evaluate each model on subgroups
subgroups = [
    (df_test_full['age_group'] == 'young', 'Young (<50)'),
    (df_test_full['age_group'] == 'middle', 'Middle (50-70)'),
    (df_test_full['age_group'] == 'elderly', 'Elderly (>70)')
]

subgroup_results = []
for model, model_name in [(model_clean, 'Clean'), 
                          (model_poisoned_label, 'Label Poisoned'),
                          (model_poisoned_feature, 'Feature Poisoned')]:
    for mask, name in subgroups:
        result = evaluate_subgroup(model, scaler, df_test_full, feature_cols, mask, name)
        if result:
            result['model'] = model_name
            subgroup_results.append(result)

df_subgroup_results = pd.DataFrame(subgroup_results)
print("\nSubgroup Performance Analysis")
print("=" * 70)
pivot = df_subgroup_results.pivot(index='subgroup', columns='model', values='recall')
pivot = pivot[['Clean', 'Label Poisoned', 'Feature Poisoned']]
print("\nRecall (Sensitivity) by Subgroup:")
print(pivot.round(3))

In [ ]:
# Visualize the attack impact
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of recall by subgroup
ax = axes[0]
pivot_plot = pivot.reset_index()
x = np.arange(len(pivot_plot))
width = 0.25

ax.bar(x - width, pivot_plot['Clean'], width, label='Clean', color=COLORS['clean'])
ax.bar(x, pivot_plot['Label Poisoned'], width, label='Label Poisoned', color=COLORS['poisoned'])
ax.bar(x + width, pivot_plot['Feature Poisoned'], width, label='Feature Poisoned', color=COLORS['detected'])

ax.set_xlabel('Age Group', fontsize=12)
ax.set_ylabel('Recall (Sensitivity)', fontsize=12)
ax.set_title('Impact of Poisoning on Subgroup Sensitivity', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(pivot_plot['subgroup'])
ax.legend(fontsize=10)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

# Annotate the targeted group
ax.annotate('TARGETED\nGROUP', xy=(2, 0.1), fontsize=10, ha='center', 
            color=COLORS['poisoned'], fontweight='bold')

# Show drop in recall for elderly
ax = axes[1]
models = ['Clean', 'Label\nPoisoned', 'Feature\nPoisoned']
elderly_recall = [pivot.loc['Elderly (>70)', 'Clean'],
                  pivot.loc['Elderly (>70)', 'Label Poisoned'],
                  pivot.loc['Elderly (>70)', 'Feature Poisoned']]

colors = [COLORS['clean'], COLORS['poisoned'], COLORS['detected']]
ax.bar(models, elderly_recall, color=colors)
ax.set_ylabel('Recall for Elderly Patients', fontsize=12)
ax.set_title('Targeted Attack Impact on Elderly Subgroup', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

# Add percentage drop annotations
baseline = elderly_recall[0]
for i, val in enumerate(elderly_recall[1:], 1):
    drop = (baseline - val) / baseline * 100
    ax.annotate(f'-{drop:.0f}%', xy=(i, val + 0.03), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n⚠️ Key Finding: Poisoning attacks disproportionately affect the targeted subgroup")
print("   while overall model metrics may appear acceptable.")

---

## 4. Detecting Poisoned Samples

### 4.1 Isolation Forest for Outlier Detection

Isolation Forest identifies samples that are easy to isolate from the rest of the data.

In [ ]:
def detect_outliers_isolation_forest(X, contamination=0.1):
    """
    Detect outliers using Isolation Forest.
    
    Returns:
    --------
    outlier_mask : boolean array (True = outlier)
    outlier_scores : anomaly scores (-1 = most anomalous)
    """
    iso_forest = IsolationForest(
        contamination=contamination,
        random_state=42,
        n_estimators=100
    )
    
    predictions = iso_forest.fit_predict(X)
    scores = iso_forest.decision_function(X)
    
    outlier_mask = predictions == -1
    
    return outlier_mask, scores

# Apply to feature-poisoned data
X_poisoned = df_poisoned_features[feature_cols].values
scaler_detect = StandardScaler()
X_poisoned_scaled = scaler_detect.fit_transform(X_poisoned)

# Detect outliers
outlier_mask, outlier_scores = detect_outliers_isolation_forest(
    X_poisoned_scaled, 
    contamination=0.05  # Expected proportion of outliers
)

# Evaluate detection performance
true_poisoned = np.zeros(len(df_poisoned_features), dtype=bool)
true_poisoned[poisoned_feature_indices] = True

detected_as_poisoned = outlier_mask

true_positives = (true_poisoned & detected_as_poisoned).sum()
false_positives = (~true_poisoned & detected_as_poisoned).sum()
false_negatives = (true_poisoned & ~detected_as_poisoned).sum()

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0

print("Isolation Forest Detection Results")
print("=" * 50)
print(f"True poisoned samples: {true_poisoned.sum()}")
print(f"Detected as outliers: {detected_as_poisoned.sum()}")
print(f"\nTrue Positives: {true_positives}")
print(f"False Positives: {false_positives}")
print(f"False Negatives: {false_negatives}")
print(f"\nPrecision: {precision:.1%}")
print(f"Recall: {recall:.1%}")

### 4.2 Local Outlier Factor (LOF)

LOF compares the local density of a sample to its neighbors, identifying samples in sparse regions.

In [ ]:
def detect_outliers_lof(X, contamination=0.1, n_neighbors=20):
    """
    Detect outliers using Local Outlier Factor.
    """
    lof = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=contamination
    )
    
    predictions = lof.fit_predict(X)
    scores = lof.negative_outlier_factor_
    
    outlier_mask = predictions == -1
    
    return outlier_mask, scores

# Apply LOF
outlier_mask_lof, scores_lof = detect_outliers_lof(
    X_poisoned_scaled, 
    contamination=0.05
)

# Evaluate
tp_lof = (true_poisoned & outlier_mask_lof).sum()
fp_lof = (~true_poisoned & outlier_mask_lof).sum()
fn_lof = (true_poisoned & ~outlier_mask_lof).sum()

precision_lof = tp_lof / (tp_lof + fp_lof) if (tp_lof + fp_lof) > 0 else 0
recall_lof = tp_lof / (tp_lof + fn_lof) if (tp_lof + fn_lof) > 0 else 0

print("\nLocal Outlier Factor Detection Results")
print("=" * 50)
print(f"Precision: {precision_lof:.1%}")
print(f"Recall: {recall_lof:.1%}")

In [ ]:
# Visualize outlier scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Isolation Forest scores
ax = axes[0]
ax.hist(outlier_scores[~true_poisoned], bins=50, alpha=0.7, 
        label='Clean samples', color=COLORS['clean'], density=True)
ax.hist(outlier_scores[true_poisoned], bins=50, alpha=0.7, 
        label='Poisoned samples', color=COLORS['poisoned'], density=True)
ax.axvline(x=np.percentile(outlier_scores, 5), color='black', linestyle='--', 
           label='Detection threshold')
ax.set_xlabel('Anomaly Score (higher = more normal)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Isolation Forest: Score Distribution', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# LOF scores
ax = axes[1]
ax.hist(scores_lof[~true_poisoned], bins=50, alpha=0.7, 
        label='Clean samples', color=COLORS['clean'], density=True)
ax.hist(scores_lof[true_poisoned], bins=50, alpha=0.7, 
        label='Poisoned samples', color=COLORS['poisoned'], density=True)
ax.set_xlabel('LOF Score (more negative = more anomalous)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Local Outlier Factor: Score Distribution', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("\n💡 Insight: Poisoned samples tend to have lower anomaly scores,")
print("   but there is overlap with clean samples, making perfect detection difficult.")

---

## 5. Robust Training Methods

### 5.1 Training with Outlier Removal

A simple defense: remove detected outliers before training.

In [ ]:
def train_with_outlier_removal(df_poisoned, feature_cols, outlier_detector='isolation_forest',
                               contamination=0.05):
    """
    Train model after removing detected outliers.
    """
    X = df_poisoned[feature_cols].values
    y = df_poisoned['sepsis'].values
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Detect outliers
    if outlier_detector == 'isolation_forest':
        outlier_mask, _ = detect_outliers_isolation_forest(X_scaled, contamination)
    else:
        outlier_mask, _ = detect_outliers_lof(X_scaled, contamination)
    
    # Remove outliers
    X_clean = X_scaled[~outlier_mask]
    y_clean = y[~outlier_mask]
    
    print(f"Removed {outlier_mask.sum()} outliers ({outlier_mask.mean():.1%})")
    print(f"Training on {len(X_clean)} samples")
    
    # Train model
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_clean, y_clean)
    
    return model, scaler, outlier_mask

# Train robust model on feature-poisoned data
model_robust, scaler_robust, removed_mask = train_with_outlier_removal(
    df_poisoned_features, 
    feature_cols,
    contamination=0.08
)

In [ ]:
# Evaluate robust model on test set
X_test_robust = scaler_robust.transform(df_test_full[feature_cols].values)
y_test_full = df_test_full['sepsis'].values

y_pred_robust = model_robust.predict(X_test_robust)
y_pred_prob_robust = model_robust.predict_proba(X_test_robust)[:, 1]

# Compare subgroup performance
results_comparison = []

for mask, name in subgroups:
    # Clean model
    y_pred_clean_sub = model_clean.predict(scaler.transform(df_test_full[mask][feature_cols].values))
    recall_clean = recall_score(df_test_full[mask]['sepsis'], y_pred_clean_sub)
    
    # Poisoned model (no defense)
    y_pred_poisoned_sub = model_poisoned_feature.predict(scaler.transform(df_test_full[mask][feature_cols].values))
    recall_poisoned = recall_score(df_test_full[mask]['sepsis'], y_pred_poisoned_sub)
    
    # Robust model
    y_pred_robust_sub = model_robust.predict(scaler_robust.transform(df_test_full[mask][feature_cols].values))
    recall_robust = recall_score(df_test_full[mask]['sepsis'], y_pred_robust_sub)
    
    results_comparison.append({
        'Subgroup': name,
        'Clean Model': recall_clean,
        'Poisoned (No Defense)': recall_poisoned,
        'Poisoned (With Defense)': recall_robust
    })

df_comparison = pd.DataFrame(results_comparison)
print("\nRecall Comparison: Effect of Robust Training")
print("=" * 70)
print(df_comparison.to_string(index=False))

In [ ]:
# Visualize defense effectiveness
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(df_comparison))
width = 0.25

ax.bar(x - width, df_comparison['Clean Model'], width, 
       label='Clean Model (Baseline)', color=COLORS['clean'])
ax.bar(x, df_comparison['Poisoned (No Defense)'], width, 
       label='Poisoned (No Defense)', color=COLORS['poisoned'])
ax.bar(x + width, df_comparison['Poisoned (With Defense)'], width, 
       label='Poisoned (With Defense)', color=COLORS['target'])

ax.set_xlabel('Subgroup', fontsize=12)
ax.set_ylabel('Recall (Sensitivity)', fontsize=12)
ax.set_title('Effect of Robust Training on Poisoning Defense', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_comparison['Subgroup'])
ax.legend(fontsize=10)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Calculate recovery
elderly_clean = df_comparison[df_comparison['Subgroup'] == 'Elderly (>70)']['Clean Model'].values[0]
elderly_poisoned = df_comparison[df_comparison['Subgroup'] == 'Elderly (>70)']['Poisoned (No Defense)'].values[0]
elderly_defended = df_comparison[df_comparison['Subgroup'] == 'Elderly (>70)']['Poisoned (With Defense)'].values[0]

drop = elderly_clean - elderly_poisoned
recovery = elderly_defended - elderly_poisoned

print(f"\n📊 Defense Effectiveness for Elderly Subgroup:")
print(f"   Performance drop from poisoning: {drop:.1%}")
print(f"   Performance recovered by defense: {recovery:.1%}")
print(f"   Recovery rate: {recovery/drop*100:.0f}% of original drop")

### 5.2 Varying Poisoning Rates

Let's examine how defense effectiveness changes with different poisoning intensities.

In [ ]:
def evaluate_defense_at_poison_rate(poison_rate, defense_contamination=0.08):
    """Evaluate model performance at different poisoning rates."""
    # Create poisoned data
    df_p, _ = feature_manipulation_attack(
        df_clean,
        target_condition=target_elderly_sepsis,
        features_to_modify=['heart_rate', 'resp_rate', 'lactate', 'wbc'],
        modification_strength=0.6,
        poison_rate=poison_rate,
        seed=int(poison_rate * 100)
    )
    
    # Train without defense
    X_train, _, y_train, _, scaler_nd = prepare_data(df_p, feature_cols)
    model_nd = LogisticRegression(random_state=42, max_iter=1000)
    model_nd.fit(X_train, y_train)
    
    # Train with defense
    model_d, scaler_d, _ = train_with_outlier_removal(
        df_p, feature_cols, contamination=defense_contamination
    )
    
    # Evaluate on elderly subgroup
    elderly_test = df_test_full[df_test_full['age_group'] == 'elderly']
    y_true = elderly_test['sepsis'].values
    
    y_pred_nd = model_nd.predict(scaler_nd.transform(elderly_test[feature_cols].values))
    y_pred_d = model_d.predict(scaler_d.transform(elderly_test[feature_cols].values))
    
    return {
        'poison_rate': poison_rate,
        'recall_no_defense': recall_score(y_true, y_pred_nd),
        'recall_with_defense': recall_score(y_true, y_pred_d)
    }

# Test different poisoning rates
poison_rates = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
rate_results = []

print("Evaluating defense at different poisoning rates...")
for rate in poison_rates:
    print(f"\nPoison rate: {rate:.0%}")
    result = evaluate_defense_at_poison_rate(rate)
    rate_results.append(result)

df_rates = pd.DataFrame(rate_results)

In [ ]:
# Visualize defense effectiveness vs poisoning rate
fig, ax = plt.subplots(figsize=(10, 6))

# Baseline (clean model)
baseline_recall = df_comparison[df_comparison['Subgroup'] == 'Elderly (>70)']['Clean Model'].values[0]
ax.axhline(y=baseline_recall, color=COLORS['clean'], linestyle='--', 
           linewidth=2, label='Clean Model Baseline')

ax.plot(df_rates['poison_rate'] * 100, df_rates['recall_no_defense'], 'o-',
        color=COLORS['poisoned'], linewidth=2, markersize=8, label='No Defense')
ax.plot(df_rates['poison_rate'] * 100, df_rates['recall_with_defense'], 's-',
        color=COLORS['target'], linewidth=2, markersize=8, label='With Defense')

ax.fill_between(df_rates['poison_rate'] * 100, 
                df_rates['recall_no_defense'], 
                df_rates['recall_with_defense'],
                alpha=0.2, color=COLORS['target'], label='Defense Benefit')

ax.set_xlabel('Poisoning Rate (%)', fontsize=12)
ax.set_ylabel('Recall for Elderly Patients', fontsize=12)
ax.set_title('Defense Effectiveness vs. Poisoning Intensity', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower left')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Insights:")
print("   1. Without defense, recall drops steadily as poisoning increases")
print("   2. Outlier removal provides significant protection at moderate poisoning rates")
print("   3. At very high poisoning rates, defense becomes less effective")
print("   4. Defense works best when poisoning is not yet the majority of target data")

---

## 6. Summary and Best Practices

### Key Takeaways

1. **Data poisoning can be highly targeted**: Attacks may degrade performance for specific subgroups while maintaining acceptable overall metrics.

2. **Label flipping is simple but effective**: Changing labels for even a small fraction of targeted samples can substantially reduce sensitivity for that group.

3. **Feature manipulation is harder to detect**: Because labels remain correct, these attacks may bypass simple validation checks.

4. **Outlier detection provides partial defense**: Methods like Isolation Forest and LOF can identify some poisoned samples, but overlap with clean data limits effectiveness.

5. **Defense effectiveness depends on poisoning rate**: At low-to-moderate poisoning rates, outlier removal significantly mitigates attacks. At very high rates, defenses struggle.

### Recommended Defense Practices

| Practice | Description | Implementation |
|----------|-------------|----------------|
| Data provenance | Track source of all training data | Audit trails, digital signatures |
| Label validation | Expert review of label quality | Random sampling, consensus labeling |
| Outlier screening | Remove statistical outliers before training | Isolation Forest, LOF |
| Subgroup monitoring | Track performance across demographic groups | Stratified evaluation metrics |
| Robust training | Use algorithms resistant to outliers | Trimmed loss functions, ensemble methods |

In [ ]:
print("\n" + "="*70)
print("NOTEBOOK COMPLETE: Data Poisoning and Robust Training")
print("="*70)
print("""
✅ Implemented label flipping and feature manipulation attacks
✅ Demonstrated targeted impact on specific patient subgroups
✅ Applied outlier detection for poisoned sample identification
✅ Evaluated robust training with outlier removal
✅ Analyzed defense effectiveness across poisoning rates

Remember: Data quality controls and provenance tracking are your first
line of defense against poisoning attacks.
""")

---

## Exercises

1. **Backdoor attack**: Implement a backdoor attack where a specific feature pattern (e.g., heart_rate > 120 AND lactate > 3) triggers misclassification. Test whether outlier detection can identify backdoor samples.

2. **Clean-label attack**: Design an attack that only modifies features, not labels. Can you craft perturbations that cause the model to learn wrong associations?

3. **Ensemble defense**: Train an ensemble of models on bootstrapped samples. Does the ensemble provide better robustness than a single model?

4. **Adaptive contamination**: Instead of fixed contamination rate, implement an adaptive threshold that adjusts based on score distribution.

5. **Real data application**: If you have access to a real healthcare dataset, apply these poisoning and defense techniques. How do results compare to simulated data?

---

*This notebook accompanies Chapter 2 of AI in Healthcare Volume 2.*